# Bij AI variant-interpretation component (GEPER) — Genetic Evaluation & Prediction Engine

This notebook installs dependencies, fetches the GEPER project (the Bij AI variant-interpretation component), and runs the full multi-model variant analysis pipeline on a VCF file, with no manual edits required.

**Runtime:** In Colab, go to `Runtime > Change runtime type` and select a GPU before running. GEPER auto-detects GPU/CPU; it will run on CPU if no GPU is available, but Evo 2 (7B) has no CPU path at all and will be skipped entirely without a GPU. **Evo 2 additionally requires a GPU with compute capability >= 8.0 (Ampere/Ada/Hopper -- e.g. A100 or L4), because its FlashAttention-2 dependency's official CUDA backend does not support Turing GPUs (compute capability 7.5), which includes the free-tier T4.** On a T4, DNABERT-2, HyenaDNA, RNA-FM, ESM-2, and AlphaMissense all still run normally -- only Evo 2 is skipped (with one clear startup warning naming the GPU and the shortfall), and the router falls back to DNABERT-2/HyenaDNA for the variants that would otherwise have used it. ESM-2 (650M) is also substantially faster and less memory-constrained on any GPU vs CPU.


## 1. Install dependencies

In [ ]:
# Core stack, pinned to the exact set validated together for GEPER (see requirements.txt for the full rationale / evidence for each pin).
# torch is pinned to 2.7.1 specifically because Evo2's flash-attn dependency below is compiled from source against this exact torch+CUDA build -- this is Arc Institute's own current documented combination (github.com/ArcInstitute/evo2), not an arbitrary choice.
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install -q 'transformers==4.56.2' 'accelerate>=1.14.0,<2.0.0' 'einops==0.8.1' 'torchvision==0.22.1' 'safetensors>=0.4.0' biopython requests pandas numpy tqdm pyyaml
!pip install -q rna-fm

# Evo2 (models/evo2.py): flash-attn has no prebuilt PyPI wheel and compiles from source against the torch build installed above -- this step is genuinely slow (several minutes) the first time.
!pip install -q flash-attn==2.8.0.post2 --no-build-isolation
!pip install -q evo2
# GEPER defaults to the 'evo2_7b_base' checkpoint (GEPER_EVO2_VARIANT), which -- unlike plain 'evo2_7b' -- does NOT need Transformer Engine/FP8/Hopper hardware; see README.md "12. Hardware notes" and https://github.com/ArcInstitute/evo2/issues/208 for why that distinction matters. Skip the two flash-attn/evo2 lines above entirely if this runtime has no GPU -- Evo2 is auto-skipped, never a hard failure, the same way every other optional model dependency is.

# AlphaMissense (models/alphamissense.py): system binary, not a pip package, and has no torch/transformers dependency.
!apt-get -qq install -y tabix

print('Core dependencies installed.')


## 1b. Install `tabix` (required for AlphaMissense)

AlphaMissense is integrated as an indexed lookup against DeepMind's precomputed prediction catalogue (no trained weights are released, so there's no model to download here) via the `tabix` CLI, a bioinformatics tool distributed as a system package, not a pip package. It is **not** the `google-deepmind/alphamissense` PyPI/GitHub package -- GEPER does not need or use that package. If `tabix` isn't installed, GEPER detects that at startup and skips the AlphaMissense stage for every variant (never a crash) -- run this cell if you want AlphaMissense results included.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tabix
print('tabix installed.')

## 2. (Optional) Persist the model cache to Google Drive

By default GEPER caches downloaded HuggingFace weights under `./model_cache` in the ephemeral Colab VM, so a disconnect means re-downloading everything. Mounting Drive and pointing `GEPER_CACHE_DIR` at it makes the cache (and thus GPU downloads) persist across sessions. Skip this cell if you don't need that.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
import os
# os.environ['GEPER_CACHE_DIR'] = '/content/drive/MyDrive/geper_model_cache'
print('Model cache dir:', os.environ.get('GEPER_CACHE_DIR', './model_cache (ephemeral)'))


## 3. Get the GEPER project onto the Colab filesystem

If you're working from a zip/GitHub export of this project, upload or clone it here. This cell assumes the project has been placed at `/content/geper`.


In [ ]:
# Option A: upload a zip of the geper/ project folder via the Colab file browser,
# then unzip it:
# from google.colab import files
# uploaded = files.upload()
# !unzip -q geper.zip -d /content/

# Option B: if GEPER lives in your own GitHub repo:
# !git clone https://github.com/<your-org>/geper.git /content/geper

import sys, os
PROJECT_ROOT = '/content/geper'
assert os.path.isdir(PROJECT_ROOT), f'GEPER project not found at {PROJECT_ROOT} — upload/clone it first.'
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('GEPER project root:', PROJECT_ROOT)


## 4. (Optional) Pre-install HyenaDNA

HyenaDNA is only used for long-context / structural variants and is **not required to run this cell** — as of this version, GEPER now auto-installs HyenaDNA (and RNA-FM) the first time they are actually needed, so `python main.py` / `pipeline.run()` works standalone with no manual setup step.

Run this cell only if you want to control *when* the one-time HyenaDNA source clone happens (e.g. to do it now, in a setup cell, rather than mid-run the first time a long-context variant is routed to it).


In [ ]:
from models.hyenadna import install_hyenadna_colab, is_hyenadna_installed

if is_hyenadna_installed():
    print('HyenaDNA already installed.')
else:
    ok = install_hyenadna_colab()
    print('HyenaDNA source installed:', ok)
    print('The checkpoint (e.g. hyenadna-medium-450k-seqlen) downloads automatically the first time HyenaDNA actually loads.')


## 5. Upload a VCF file to analyze

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your .vcf or .vcf.gz file
vcf_filename = list(uploaded.keys())[0]
print('Uploaded:', vcf_filename)


## 6. Test run first: `max_variants`

Large VCFs (e.g. millions of variants) are far too slow for a first sanity check. Run against just the first handful of variants to confirm everything loads and executes end-to-end before committing to a full run.


In [ ]:
from pipeline.orchestrator import GeperPipeline

pipeline = GeperPipeline(
    blast_mode='remote',       # 'local' if you have BLAST+ and a local DB configured
    species='human',
    assembly=None,             # e.g. 'GRCh38'; None lets GEPER auto-detect from the VCF header
    output_dir='/content/geper_output',
)

test_result = pipeline.run(vcf_filename, max_variants=20)
print(f"Test run processed {test_result['variant_count']} variant(s). Check the summary log above.")


## 7. Full run

Once the test run above looks right, run against the whole file. This resumes automatically from `output_dir/geper_results.json` if a previous run (e.g. before a Colab disconnect) was already partially completed there -- pass `resume=False` to force starting over, or point `output_dir` elsewhere for a clean run.


In [ ]:
result_document = pipeline.run(vcf_filename)
print(f"Processed {result_document['variant_count']} variant(s) total.")


## 8. View the JSON output

In [ ]:
import json
print(json.dumps(result_document['variants'][0], indent=2)[:3000], '...')


## 9. View the human-readable Markdown report

In [ ]:
from IPython.display import Markdown, display
with open('/content/geper_output/geper_report.md') as f:
    report_md = f.read()
display(Markdown(report_md))


## 10. Download results

In [ ]:
from google.colab import files
files.download('/content/geper_output/geper_results.json')
files.download('/content/geper_output/geper_report.md')
